# GridSight UK — Train on Google Colab (GPU)

This notebook trains and evaluates three probabilistic solar power forecasting models:
- Model A: **Primary Stacking Model** (TCN-Q + LGBM-Q ➔ Linear-Q Stacking Stack)
- Model B: **Standalone LSTM-Q Model** (Deep LSTM Sequence Forecaster)
- Model C: **Pretrained Chronos-Q Model** (Zero-shot Foundation Model Forecaster)

Please run each cell sequentially. **No GitHub account or local git setup is required** — simply upload `modeling.zip` and download Gold features from HuggingFace.

> **CRITICAL PRE-REQUISITE**: Ensure your runtime is GPU-accelerated: **Runtime ➔ Change runtime type ➔ T4 GPU ➔ Save**.

## 1. Verify GPU Availability

In [ ]:
!nvidia-smi -L
import torch
print('PyTorch Version:', torch.__version__, '| CUDA active:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU not active! Please select Runtime -> Change runtime type -> T4 GPU'

## 2. Upload Code to Colab (upload `modeling.zip`)

On your **local machine**, create a zip archive of the model directories (`modeling/`, `lstm_q/`, and `chronos_q/`):
```bash
zip -r modeling.zip modeling lstm_q chronos_q
```
Then run the cell below and upload the resulting `modeling.zip` file.

In [ ]:
from google.colab import files
uploaded = files.upload()                 # select and upload modeling.zip
!unzip -oq modeling.zip -d .
!ls -d modeling lstm_q chronos_q
print('SUCCESS: Code directories modeling/, lstm_q/, and chronos_q/ extracted.')

## 3. Install Package Dependencies

Google Colab pre-installs **PyTorch (with GPU capabilities)**, so we do not reinstall PyTorch. We install the other required packages and dependencies for the Chronos foundation model.

> **IMPORTANT**: You must run this cell first. If you encounter any `ModuleNotFoundError` when importing modules later, please restart your session: click **Runtime ➔ Restart session** in the top menu, then continue running the subsequent cells.

In [ ]:
!pip install lightgbm==4.6.0 scikit-learn==1.9.0 joblib==1.5.3 huggingface_hub loguru==0.7.3 matplotlib==3.10.9 mlflow==3.14.0 optuna==4.9.0 pandera==0.32.1 "chronos-forecasting>=1.5.0" "transformers>=4.48,<5" "accelerate>=0.32"

## 4. Download Gold Features from HuggingFace
Provide your **HuggingFace token** (with read access to the team's repositories) when prompted to pull the Gold features.

In [ ]:
from getpass import getpass
from huggingface_hub import snapshot_download
tok = getpass('HF token: ')
snapshot_download('gridsight-team/gridsight-gold', repo_type='dataset',
                  local_dir='data/gold', token=tok)
!echo 'Total Gold parquet files downloaded:' && find data/gold -name '*.parquet' | wc -l

## 5. Smoke Test (Runs quickly in ~1 minute)
Execute this smoke test to verify all models and data loaders run correctly without syntax or hardware compatibility issues.

In [ ]:
!python -m modeling --fast
!python -m lstm_q.train --fast
!python -m chronos_q --fast --gold-dir data/gold/gold_features

## 6. Train & Run the Models (Fully on GPU)
The training and execution steps have been separated into individual cells so you can run, log, and monitor each model independently.

### Step 6.1: Tune Hyperparameters for LSTM (Model B)
Uses Optuna random search to find optimal hyperparameters for the PyTorch LSTM network.

In [ ]:
!python -m lstm_q.tune --trials 5 --epochs 5

### Step 6.2: Train Model A (Stacking Pipeline)
Trains the TCN-Q neural model and LGBM-Q tabular models across OOF folds, then fits the Linear-Q stacker meta-learner.

In [ ]:
!python -m modeling --epochs 40

### Step 6.3: Train Model B (Standalone LSTM-Q)
Trains the standalone PyTorch LSTM sequence model using the best tuned parameters from the tuning step.

In [ ]:
!python -m lstm_q.train --epochs 40

### Step 6.4: Run Model C (Pretrained Chronos-Q Zero-Shot Inference & Calibration)
Since Chronos is a pre-trained time-series foundation model, **no training is needed**. Running this command performs zero-shot prediction and calculates post-hoc calibration factors directly. This runs extremely quickly (~1-2 minutes).

In [ ]:
!python -m chronos_q --horizon-steps 48 --gold-dir data/gold/gold_features

## 7. Model Evaluation and Visualization
Generate evaluation reports, fan charts (visualizing the calibrated 80% prediction interval and median forecast vs. actual generation), and calibration dashboards for all three models.

In [ ]:
# Evaluate Model A (Stacking)
!python -m modeling.evaluate --split test
from IPython.display import Image, display
print("=== Model A (Stacking) Fan Chart ===")
display(Image('artifacts/model/plots/evaluation_fan.png'))
print("=== Model A (Stacking) Dashboard ===")
display(Image('artifacts/model/plots/evaluation_dashboard.png'))

In [ ]:
# Evaluate Model B (Standalone LSTM-Q)
!python -m lstm_q.evaluate --split test
from IPython.display import Image, display
print("=== Model B (LSTM-Q) Fan Chart ===")
display(Image('artifacts/lstm/plots/evaluation_fan.png'))
print("=== Model B (LSTM-Q) Dashboard ===")
display(Image('artifacts/lstm/plots/evaluation_dashboard.png'))

In [ ]:
# Evaluate Model C (Pretrained Chronos-Q)
!python -m chronos_q.evaluate --split test --artifacts artifacts/chronos
from IPython.display import Image, display
print("=== Model C (Chronos-Q) Fan Chart ===")
display(Image('artifacts/chronos/plots/evaluation_fan.png'))
print("=== Model C (Chronos-Q) Dashboard ===")
display(Image('artifacts/chronos/plots/evaluation_dashboard.png'))

## 8. Download Trained Models & Artifacts
Compare metric summaries, compress the entire artifacts directory (containing models, metadata, metrics, and plots), and download it locally.

In [ ]:
import json
print("=== Model A (Stacking) Metrics ===")
print(json.dumps(json.load(open('artifacts/model/metrics.json')), indent=2))
print("=== Model B (LSTM-Q) Metrics ===")
print(json.dumps(json.load(open('artifacts/lstm/metrics.json')), indent=2))
print("=== Model C (Chronos-Q) Metrics ===")
print(json.dumps(json.load(open('artifacts/chronos/metrics.json')), indent=2))

!zip -rq artifacts.zip artifacts
from google.colab import files
files.download('artifacts.zip')   # contains model/, lstm/, and chronos/ sub-folders

### (Optional) Backup to Google Drive
Mount your Google Drive to persistently save your artifacts directory so it survives session timeout.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r artifacts /content/drive/MyDrive/gridsight_artifacts

---
### Data Splitting (Chronological, no shuffling):
- **Train Set**: `< 2024-07-01` (~2023 + H1 2024)
- **Validation Set**: `2024-07-01` to `2024-09-30`
- **Test Set**: `2024-10-01` to `2024-12-31`
- Out-Of-Fold (OOF) cross-validation folds are created within the training set to construct stacked features for the Linear-Q stacker without leakage.

### Performance Metrics:
- **Pinball Loss**: Average pinball loss across q10, q50, q90. Lower is better.
- **Coverage (PICP)**: Percentage of true observations falling between the q10 and q90 prediction intervals (success gate: `[0.78, 0.82]`).
- **Crossing Rate**: Percentage of slots violating $q_{10} \le q_{50} \le q_{90}$ (success gate: `0.00%`).
- **Skill Score vs NESO q50**: Improvement percentage over the NESO operator solar forecast (> 0 means outperforming NESO).